# Infinite Museum — LoRA Fine-tune
**Model:** Qwen/Qwen2.5-7B-Instruct  
**Data:** world_bible_train.jsonl (317 examples)  
**Method:** LoRA via Unsloth + TRL SFTTrainer

### Steps
1. Install dependencies
2. Upload training data
3. Load model
4. Train
5. Save adapter to Google Drive

## Step 1 — Check GPU

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## Step 2 — Install dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate datasets huggingface_hub

## Step 3 — Upload training file
Run this cell and upload your `world_bible_train.jsonl` from:  
`C:\Users\vishn\OneDrive\Desktop\infinite-museum\data\synthetic\world_bible_train.jsonl`

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload world_bible_train.jsonl

import os
TRAIN_FILE = list(uploaded.keys())[0]
print(f"Uploaded: {TRAIN_FILE}")

# Quick sanity check
import json
with open(TRAIN_FILE) as f:
    lines = f.readlines()
print(f"Total training examples: {len(lines)}")
print("Sample keys:", list(json.loads(lines[0]).keys()))

## Step 4 — Load model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
print("Model loaded.")

## Step 5 — Attach LoRA adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("LoRA adapter attached.")

## Step 6 — Prepare dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=TRAIN_FILE, split="train")

def format_row(row):
    return {
        "text": tokenizer.apply_chat_template(
            row["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

dataset = dataset.map(format_row)
print(f"Dataset ready: {len(dataset)} examples")
print("Sample text (first 300 chars):")
print(dataset[0]["text"][:300])

## Step 7 — Train

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/content/infinite-museum-adapter"

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=5,
        save_strategy="epoch",
        output_dir=OUTPUT_DIR,
        optim="paged_adamw_8bit",
        bf16=True,
        report_to="none",
    ),
)

print("Starting training...")
trainer.train()
print("Training complete!")

## Step 8 — Save adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

import os
files_saved = os.listdir(OUTPUT_DIR)
print("Files:", files_saved)

## Step 9 — Save to Google Drive (so you don't lose it when Colab disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_PATH = "/content/drive/MyDrive/infinite-museum-adapter"
shutil.copytree(OUTPUT_DIR, DRIVE_PATH, dirs_exist_ok=True)
print(f"Adapter backed up to Google Drive at: {DRIVE_PATH}")

## Step 10 — Optional: Push to HuggingFace Hub

In [ ]:
# Fill in your details and run this cell to push the adapter to HuggingFace
HF_TOKEN = "hf_your_token_here"   # replace with your token
HF_REPO  = "your-username/infinite-museum-qwen7b-lora"  # replace with your repo

from huggingface_hub import login
login(token=HF_TOKEN)

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"Pushed to https://huggingface.co/{HF_REPO}")